# Delta Air Lines Quarterly Revenue Forecast

An OLS time-series model that forecasts Delta Air Lines (NYSE: DAL) quarterly total revenue. A linear time trend is combined with COVID and Q1-seasonality dummies (and their interactions with time), fit on a 75% training split, validated on the 25% hold-out, and used to project the next four quarters (2026 Q3 – 2027 Q2).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

pd.options.display.float_format = '{:.2f}'.format

## Load and clean

The raw file has one row per fiscal quarter. Two cleaning steps: convert the `FQ# YYYY` labels to calendar quarter-end dates, and strip the `b` (billions) suffix so revenue is numeric.

In [ ]:
delta_sales = pd.read_csv('../data/DELTA_Q_Revenue.csv')
delta_sales.rename(columns={'Delta Air Lines, Inc. (NYSE:DAL) - Total Revenue': 'delta_revenue'}, inplace=True)
display(delta_sales.head())

In [ ]:
def fq_to_date(fq_str):
    # 'FQ3 2016' -> quarter-end Timestamp
    quarter_label, year_str = fq_str.split(' ')
    year = int(year_str)
    if quarter_label.startswith('FQ') and len(quarter_label) == 3:
        quarter = int(quarter_label[2])
    else:
        return pd.NaT
    end_month_day = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}
    if quarter in end_month_day:
        return pd.Timestamp(year, *end_month_day[quarter])
    return pd.NaT

delta_sales['Dates'] = delta_sales['Dates'].astype(str).apply(fq_to_date)
delta_sales['delta_revenue'] = delta_sales['delta_revenue'].str.replace('b', '', regex=False).astype(float)
display(delta_sales.head())

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(delta_sales['Dates'], delta_sales['delta_revenue'], marker='o')
plt.title('Delta Air Lines — Quarterly Total Revenue')
plt.xlabel('Quarter')
plt.ylabel('Revenue ($B)')
plt.grid(True)
plt.show()

## Feature construction

- **`time`** — sequential period index (1…N), the linear trend.
- **`covid_dv`** — 1 for quarters in the COVID window (2020-03-11 to 2023-05-05), else 0.
- **`q1_dv`** — 1 for calendar-Q1 quarters, capturing Delta's seasonal Q1 softness.
- Each dummy is interacted with `time` to allow a different trend slope during those periods.

In [ ]:
# Linear trend: one period per row
delta_sales['time'] = range(1, len(delta_sales) + 1)

# COVID dummy + interaction
covid_start = pd.to_datetime('2020-03-11')
covid_end   = pd.to_datetime('2023-05-05')
delta_sales['covid_dv'] = np.where(
    (delta_sales['Dates'] >= covid_start) & (delta_sales['Dates'] <= covid_end), 1, 0)
delta_sales['covid_dv_interaction'] = delta_sales['time'] * delta_sales['covid_dv']

# Q1 seasonality dummy + interaction
delta_sales['q1_dv'] = np.where(delta_sales['Dates'].dt.quarter == 1, 1, 0)
delta_sales['q1_dv_interaction'] = delta_sales['time'] * delta_sales['q1_dv']

display(delta_sales.head())

## Train / test split

A 75/25 chronological split — no shuffling, since the goal is to forecast forward. The model trains on the earlier quarters and is scored on the most recent 25%.

In [ ]:
split = int(0.75 * len(delta_sales))
dt4training = delta_sales[:split]
dt4testing  = delta_sales[split:]
print(f'Train: {len(dt4training)} quarters | Test: {len(dt4testing)} quarters')

## Model

OLS on revenue levels ($B) with the trend, both dummies, and both interaction terms.

In [ ]:
features = ['time', 'covid_dv', 'covid_dv_interaction', 'q1_dv', 'q1_dv_interaction']

X_train = sm.add_constant(dt4training[features])
y_train = dt4training['delta_revenue']

model = sm.OLS(y_train, X_train).fit()
print(model.summary())

## Validation (hold-out)

Scored on the 25% hold-out using the full feature set. RMSE is in $B; MAPE is the average percentage error.

In [ ]:
X_test = sm.add_constant(dt4testing[features], has_constant='add')
y_test = dt4testing['delta_revenue']
pred_test = model.predict(X_test)

rmse = np.sqrt(np.mean((y_test - pred_test) ** 2))
mape = np.mean(np.abs((y_test - pred_test) / y_test)) * 100
print(f'Hold-out RMSE: {rmse:.2f} ($B)')
print(f'Hold-out MAPE: {mape:.2f}%')

comparison = dt4testing[['Dates', 'time']].copy()
comparison['actual'] = y_test.values
comparison['predicted'] = pred_test.values
comparison['error'] = comparison['actual'] - comparison['predicted']
display(comparison)

## Forecast — 2026 Q3 to 2027 Q2

In [ ]:
future_dates = [pd.Timestamp('2026-09-30'), pd.Timestamp('2026-12-31'),
                pd.Timestamp('2027-03-31'), pd.Timestamp('2027-06-30')]
future = pd.DataFrame({'Dates': future_dates})

last_time = delta_sales['time'].max()
future['time'] = range(last_time + 1, last_time + 1 + len(future))
future['covid_dv'] = 0  # COVID window is over
future['covid_dv_interaction'] = future['time'] * future['covid_dv']
future['q1_dv'] = np.where(future['Dates'].dt.quarter == 1, 1, 0)
future['q1_dv_interaction'] = future['time'] * future['q1_dv']

X_future = sm.add_constant(future[features], has_constant='add')
future['predicted_revenue'] = model.predict(X_future)

print('Predicted quarterly revenue ($B):')
display(future[['Dates', 'time', 'predicted_revenue']])